# PLP Data Analysis Assignment 2: Rainfall Advisory

**Analyst:** Jessca Waithera Gichangi

This notebook cleans monthly rainfall data for five counties, analyses rainfall patterns with NumPy, and prepares a planting advisory for a farmer-support NGO.

**Rainfall unit:** millimetres (mm).


## Task 1: Clean the Machakos rainfall data

The cleaning function converts valid text entries to integers and catches the specific `ValueError` raised by invalid or empty values. Invalid months are replaced with the rounded mean of valid Machakos readings.


In [1]:
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

def clean_machakos(raw_values, month_names):
    """Convert Machakos rainfall strings to integers and impute invalid months."""
    valid_values = []
    invalid_indices = []

    for index, value in enumerate(raw_values):
        try:
            valid_values.append(int(value.strip()))
        except ValueError:
            invalid_indices.append(index)

    imputed_value = round(sum(valid_values) / len(valid_values))
    cleaned_values = []
    valid_position = 0

    for index in range(len(raw_values)):
        if index in invalid_indices:
            cleaned_values.append(imputed_value)
        else:
            cleaned_values.append(valid_values[valid_position])
            valid_position += 1

    imputed_months = [month_names[index] for index in invalid_indices]
    print(f"Machakos imputed value: {imputed_value} mm")
    print("Months imputed:", ", ".join(imputed_months))
    return cleaned_values

machakos_clean = clean_machakos(machakos_raw, months)
print("Cleaned Machakos row:", machakos_clean)


Machakos imputed value: 68 mm
Months imputed: April, September
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## Task 2: Combine the county data into a NumPy array

The four supplied numeric rows and the cleaned Machakos row are combined into one 5 × 12 array.


In [2]:
all_rows = clean_rows + [machakos_clean]
R = np.array(all_rows)

print("Combined rainfall array:")
print(R)
print("Array shape:", R.shape)
print("Array dtype:", R.dtype)


Combined rainfall array:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Array shape: (5, 12)
Array dtype: int64


## Task 3: Annual totals and monthly averages

NumPy axis operations calculate each county's annual total and the average rainfall for each calendar month across all five counties.


In [3]:
annual_totals = R.sum(axis=1)
monthly_averages = R.mean(axis=0)

print("Annual rainfall totals (mm):")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total:.0f} mm")

print("\nAverage rainfall by month across all counties (mm):")
for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.1f} mm")


Annual rainfall totals (mm):
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

Average rainfall by month across all counties (mm):
January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm


## Task 4: Wettest month for each county

`np.argmax` with `axis=1` finds the index of the wettest month in each county's row. `zip()` pairs each county with its result.


In [4]:
wettest_indices = np.argmax(R, axis=1)
wettest_months = [months[index] for index in wettest_indices]

print("Wettest month by county:")
for county, wettest_month in zip(counties, wettest_months):
    print(f"{county}: {wettest_month}")


Wettest month by county:
Nairobi: April
Nakuru: April
Kisumu: April
Mombasa: May
Machakos: November


## Task 5: Count dry months

A dry month is defined as a month with rainfall below 50 mm. A boolean mask identifies these values, and summing the mask across each row gives the count per county.


In [5]:
dry_mask = R < 50
dry_month_counts = dry_mask.sum(axis=1)

print("Number of dry months (below 50 mm):")
for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count}")


Number of dry months (below 50 mm):
Nairobi: 4
Nakuru: 3
Kisumu: 0
Mombasa: 2
Machakos: 6


## Task 6: Identify the recommended two-month planting window

Add each month's rainfall to the following month's rainfall. `np.argmax` with `axis=1` identifies the consecutive two-month period with the highest combined rainfall for each county.


In [6]:
two_month_totals = R[:, :-1] + R[:, 1:]
planting_start_indices = np.argmax(two_month_totals, axis=1)
planting_windows = [
    f"{months[index]}–{months[index + 1]}"
    for index in planting_start_indices
]

print("Recommended two-month planting windows:")
for county, window in zip(counties, planting_windows):
    print(f"{county}: {window}")


Recommended two-month planting windows:
Nairobi: April–May
Nakuru: April–May
Kisumu: April–May
Mombasa: April–May
Machakos: November–December


## Task 7: Export the rainfall summary CSV

The CSV is written with Python's built-in `csv` module and contains exactly one data row per county.


In [7]:
import csv

csv_filename = "rainfall_summary.csv"
with open(csv_filename, "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "county", "annual_total_mm", "wettest_month",
        "dry_months", "planting_window"
    ])
    for row in zip(counties, annual_totals, wettest_months,
                   dry_month_counts, planting_windows):
        writer.writerow([
            row[0], int(row[1]), row[2], int(row[3]), row[4]
        ])

print(f"Created {csv_filename}")


Created rainfall_summary.csv


## Planting Advisory

The rainfall totals indicate that planting windows should be planned county by county rather than using one date for all locations. The strongest consecutive two-month rainfall periods in this dataset are the recommended starting windows shown in the summary table and CSV. Farmers should prepare land and source suitable seed ahead of those windows, while checking current local forecasts before planting. Counties with several months below 50 mm should consider water conservation, moisture retention and drought-tolerant crop options. The NGO should communicate these windows as guidance, not guarantees, because monthly historical totals do not show the timing or intensity of individual rain events.


## Data Quality Notes

The Machakos raw row contained two invalid entries: `N/A` for April and an empty value for September. The ten valid values have a mean of 67.5 mm, rounded to 68 mm; that value was imputed for both missing months. This simple mean imputation keeps the array complete but may not represent actual rainfall in either month. The advisory is therefore subject to the quality of the station records and should be checked against verified observations and current forecasts before being used for operational planting decisions.
